# Step 1: Satellite Environmental Baseline (GEE Backend)

### Rotterdam


In [2]:
import ee
import geemap
import os

# 1. Initialize Earth Engine with your group's project ID
# Replace this with your actual authorized cloud project name
PROJECT_ID = 'applied-spatial-rotterdam' 
ee.Initialize(project=PROJECT_ID)

# 2. Define the geographic window for Rotterdam (Bounding Box)
# Format: [xmin, ymin, xmax, ymax]
bbox = [4.30, 51.85, 4.60, 51.99]
rotterdam_aoi = ee.Geometry.Rectangle(bbox)

# 3. Load Landsat 8 Surface Reflectance & Filter for July/August 2025
# GEE handles the catalog filtering directly on their servers
landsat_collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                      .filterBounds(rotterdam_aoi)
                      .filterDate('2025-05-01', '2025-09-30')
                      .filter(ee.Filter.lt('CLOUD_COVER', 40))) # 40% threshold for stable coverage



# 4. Cloud Masking Function using the Landsat Quality Assessment band
def mask_L8_clouds(image):
    qa = image.select('QA_PIXEL')
    # Bit 3 = Cloud, Bit 5 = Cloud Shadow (0 means clear sky)
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    shadow_mask = qa.bitwiseAnd(1 << 5).eq(0)
    return image.updateMask(cloud_mask.And(shadow_mask))

processed_collection = landsat_collection.map(mask_L8_clouds)

# 5. Calculate Land Surface Temperature (LST) and NDVI
def calculate_metrics(image):
    # Convert Thermal Band 10 digital numbers to Celsius
    # Official USGS Formula: (DN * 0.00341802 + 149.0) - 273.15
    lst_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    
    # Calculate Vegetation Index (NDVI)
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI') # Band 5=NIR, Band 4=Red
    
    return image.addBands([lst_celsius, ndvi])

# Map the metrics function and take the seasonal mean average, then clip to our box
metrics_collection = processed_collection.map(calculate_metrics)
summer_mean = metrics_collection.mean().clip(rotterdam_aoi)

print(f"Successfully processed {processed_collection.size().getInfo()} cloud-masked scenes via GEE backend.")

# =========================================================================
# GEE INTERACTIVE VISUALIZATION (No Matplotlib/Xarray errors)
# =========================================================================

# Initialize an interactive geemap centered over Rotterdam
Map = geemap.Map(center=[51.92, 4.48], zoom=12)

# Visualization settings for Land Surface Temperature (Blue = Cool, Red = Hot)
lst_vis = {
    'bands': ['LST'],
    'min': 18,
    'max': 35,
    'palette': ['#313695', '#4575b4', '#abd9e9', '#fee090', '#f46d43', '#d73027', '#a50026']
}

# Visualization settings for Vegetation (NDVI) (Yellow = Low vegetation, Green = High vegetation)
ndvi_vis = {
    'bands': ['NDVI'],
    'min': 0.0,
    'max': 0.6,
    'palette': ['#ffffe5', '#f7fcb9', '#addd8e', '#41ab5d', '#238443', '#005a32']
}

# Add layers to the interactive map canvas
Map.addLayer(summer_mean.select('LST'), lst_vis, 'Mean Summer LST 2025 (°C)')
Map.addLayer(summer_mean.select('NDVI'), ndvi_vis, 'Mean Summer NDVI 2025')

# Add a text colorbar legend for the LST layer
# New updated syntax
# Pass the vis_params dictionary directly without using 'palette=' or 'colors=' keywords
Map.add_colorbar(vis_params=lst_vis, label="Land Surface Temperature (°C)")

# Render the interactive map window inside Jupyter Lab
Map

Successfully processed 6 cloud-masked scenes via GEE backend.


Map(center=[51.92, 4.48], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

### Guangzhou

In [3]:
import ee
import geemap
import os

# 1. Initialize Earth Engine with your group's project ID
PROJECT_ID = 'applied-spatial-rotterdam' 
ee.Initialize(project=PROJECT_ID)

# 2. Define the geographic window for Guangzhou (Bounding Box)
bbox_gz = [112.90, 22.40, 114.05, 23.95] 
guangzhou_aoi = ee.Geometry.Rectangle(bbox_gz)

# 3. Load Landsat 8 Surface Reflectance & Filter for Guangzhou AOI
# Set cloud cover threshold to 50% to maximize data pool for the mosaic engine
landsat_collection = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                      .filterBounds(guangzhou_aoi)
                      .filterDate('2025-06-01', '2025-09-30')
                      .filter(ee.Filter.lt('CLOUD_COVER', 50)))

# 4. Cloud Masking Function
def mask_L8_clouds(image):
    qa = image.select('QA_PIXEL')
    # Bit 3 = Cloud, Bit 5 = Cloud Shadow (0 means clear sky)
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0)
    shadow_mask = qa.bitwiseAnd(1 << 5).eq(0)
    return image.updateMask(cloud_mask.And(shadow_mask))

processed_collection = landsat_collection.map(mask_L8_clouds)

# Print tracking size BEFORE the collection properties are modified by the mosaic function
print(f"Successfully processed {processed_collection.size().getInfo()} cloud-masked scenes via GEE backend.")

# 5. Calculate Land Surface Temperature (LST) and NDVI per scene
def calculate_metrics(image):
    lst_celsius = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return image.addBands([lst_celsius, ndvi])

# Map the calculations over the collection
metrics_collection = processed_collection.map(calculate_metrics)

# 6. Quality Mosaic Compilation (Fills the holes using maximum greenness index)
summer_mean = metrics_collection.qualityMosaic('NDVI').clip(guangzhou_aoi)
print("Seamless summer mosaic successfully compiled!")

# =========================================================================
# GEE INTERACTIVE VISUALIZATION
# =========================================================================

# Initialize the Map centered over Guangzhou
Map = geemap.Map(center=[23.13, 113.26], zoom=10)

# Visualization settings (Subtropical cities run hotter, adjusted max scale to 42°C)
lst_vis = {
    'bands': ['LST'],
    'min': 22,
    'max': 42, 
    'palette': ['#313695', '#4575b4', '#abd9e9', '#fee090', '#f46d43', '#d73027', '#a50026']
}

ndvi_vis = {
    'bands': ['NDVI'],
    'min': 0.0,
    'max': 0.6,
    'palette': ['#ffffe5', '#f7fcb9', '#addd8e', '#41ab5d', '#238443', '#005a32']
}

# Add layers to the interactive map canvas
Map.addLayer(summer_mean.select('LST'), lst_vis, 'Guangzhou Mean Summer LST 2025 (°C)')
Map.addLayer(summer_mean.select('NDVI'), ndvi_vis, 'Guangzhou Mean Summer NDVI 2025')

# Add colorbar using proper dictionary mapping syntax
Map.add_colorbar(vis_params=lst_vis, label="Land Surface Temperature (°C)")

# Render map window inside notebook
Map

Successfully processed 30 cloud-masked scenes via GEE backend.
Seamless summer mosaic successfully compiled!


Map(center=[23.13, 113.26], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…